本教程介绍UniParser Tools的高阶用法。

本文以 30001 端口提供的UniParser服务为例进行演示。

该端口服务提供最新Parser能力支持，可以提取
- 图片+图题+图注
- 表格+表题+表注
- 分子+分子索引
- 公式+公式索引

## 导入依赖

In [ ]:
import json
import os

from IPython.display import HTML, Latex, Markdown, display

from uniparser_tools.api.clients import UniParserClient
from uniparser_tools.common.constant import FormatFlag, ParseMode, ParseModeTextual
from uniparser_tools.utils.convert import dict2obj
from uniparser_tools.utils.processor import tree_repr


## 初始化

In [ ]:
# 目前有多个可用host开启，但是不同host对应功能不完全相同，解析质量也不一样
# 具体请在售后群中咨询相关的的host信息和功能
# 使用时请勿并发数量过大，公开Uni-Parser服务最高仅允许5并发

host = "https://uniparser.dp.tech/"  # 官网

# 替换为你的认证 api key
api_key = os.getenv('UNIPARSER_API_KEY')

# 初始化客户端
parser = UniParserClient(host=host, api_key=api_key)

# 创建一个目录来保存解析结果
save_dir = "./outputs/advance"
os.makedirs(save_dir, exist_ok=True)

## 解析文件

In [ ]:
# 设置解析文件路径
pdf_path = "./tasks/He_Deep_Residual_Learning_CVPR_2016_paper.pdf"

# 提交解析任务
trigger_file_result = parser.trigger_file(
    pdf_path,
    textual=ParseModeTextual.DigitalExported,
    table=ParseMode.OCRFast,
    molecule=ParseMode.OCRFast,
    chart=ParseMode.DumpBase64,
    figure=ParseMode.DumpBase64,
    expression=ParseMode.DumpBase64,
    equation=ParseMode.OCRFast,
)
if trigger_file_result["status"] != "success":
    print(json.dumps(trigger_file_result, indent=4))
    raise Exception("trigger file failed")
print(f"trigger file success, token is: {trigger_file_result['token']}")

## 获取结果
> 可以持有token多次进行获取

In [ ]:
assert trigger_file_result["status"] == "success"
token = trigger_file_result["token"]
# token = "<TASK_TOKEN>"
formatted = FormatFlag.Plain  # formatted 只对objects和content产生作用，pages_dict和pages_tree不受影响
result = parser.get_formatted(
    token,
    content=False,
    objects=False,
    pages_dict=False,
    pages_tree=True,
    molecule_source=False,
    textual=formatted,
    chart=formatted,
    table=formatted,
    molecule=formatted,
    equation=formatted,
    figure=formatted,
    expression=formatted,
)
if result["status"] != "success":
    print(f"Get formatted failed for {formatted}, results is: {json.dumps(result, indent=4)}")


### 1. 重建文档结构树

In [ ]:
pages_tree = dict2obj(result["pages_tree"]) 

### 2. 查看文档结构树
> 服务更新或升级可能会导致后文代码需要做细微调整

In [ ]:
pages_tree[0]

### 3. 展示全文

In [ ]:
display(Markdown("\n\n".join([item.format_as(FormatFlag.Markdown) for item in pages_tree[0]])))

#### - 展示图文对

In [ ]:
img_cap_group = pages_tree[0][8]

In [ ]:
print(tree_repr(img_cap_group))

In [ ]:
one_fig = img_cap_group.items[0].items[0]

display(HTML(f'<img src="data:image/png;base64,{one_fig.source}" />'))

In [ ]:
display(HTML(one_fig.format_as(FormatFlag.Html)))

In [ ]:
display(HTML(img_cap_group.format_as(FormatFlag.Html)))

In [ ]:
display(Markdown(img_cap_group.format_as(FormatFlag.Markdown)))

In [ ]:
# LaTeX 不支持 base64 图片，理论上支持图片文件，但目前未测试
display(Latex(img_cap_group.format_as(FormatFlag.Latex)))

#### - 展示公式

In [ ]:
math_group = pages_tree[2][14]

In [ ]:
print(tree_repr(math_group))

In [ ]:
display(Markdown(math_group.format_as(FormatFlag.Markdown)))

#### - 展示表格

In [ ]:
table_group = pages_tree[5][0]

In [ ]:
print(tree_repr(table_group))

In [ ]:
display(Markdown(table_group.format_as(FormatFlag.Markdown)))

### 4. 输出到文件

In [ ]:
with open(f"{save_dir}/page_5.md", "w") as f:
    f.write("\n\n".join([item.format_as(FormatFlag.Markdown) for item in pages_tree[5]]))